# 04 — Asian Option Pricing

Asian options have a payoff that depends on the **average** price of the
underlying over a specified period. This notebook covers:

1. Arithmetic vs geometric averaging
2. Analytical (BSM) closed-form
3. Binomial tree pricing
4. Monte Carlo pricing
5. Forward-starting Asians
6. Explicit fixing dates
7. Seasoned (in-progress) Asians
8. American exercise

## 1) Setup

In [4]:
import datetime as dt

import derivatives_pricing as dp

In [5]:
pricing_date = dt.datetime(2025, 6, 15)
maturity = dt.datetime(2026, 6, 15)
r = 0.05

curve = dp.DiscountCurve.flat(rate=r, end_time=2.0)
market = dp.MarketData(pricing_date=pricing_date, discount_curve=curve, currency="USD")
underlying = dp.UnderlyingData(initial_value=100.0, volatility=0.20, market_data=market)

## 2) Arithmetic Asian — Analytical (BSM)

The analytical engine uses the Turnbull-Wakeman moment-matching approximation
for arithmetic average options.

In [6]:
arith_call = dp.AsianSpec(
    averaging=dp.AsianAveraging.ARITHMETIC,
    exercise_type=dp.ExerciseType.EUROPEAN,
    option_type=dp.OptionType.CALL,
    strike=100.0,
    maturity=maturity,
    num_observations=252,  # daily averaging
)

bsm_arith = dp.OptionValuation(
    underlying=underlying, spec=arith_call, pricing_method=dp.PricingMethod.BSM
)
print(f"Arithmetic Asian call (BSM): {bsm_arith.present_value():.4f}")

Arithmetic Asian call (BSM): 5.7789


## 3) Geometric Asian — Analytical (BSM)

Geometric average options have an exact closed-form solution.

In [7]:
geo_call = dp.AsianSpec(
    averaging=dp.AsianAveraging.GEOMETRIC,
    exercise_type=dp.ExerciseType.EUROPEAN,
    option_type=dp.OptionType.CALL,
    strike=100.0,
    maturity=maturity,
    num_observations=252,
)

bsm_geo = dp.OptionValuation(
    underlying=underlying, spec=geo_call, pricing_method=dp.PricingMethod.BSM
)
print(f"Geometric Asian call (BSM): {bsm_geo.present_value():.4f}")

# The geometric average is always <= arithmetic average, so the geometric
# option price is always <= arithmetic option price for calls.
print(f"\nArithmetic - Geometric: {bsm_arith.present_value() - bsm_geo.present_value():.4f}")

Geometric Asian call (BSM): 5.5417

Arithmetic - Geometric: 0.2372


## 4) Monte Carlo Pricing

In [8]:
sim_config = dp.SimulationConfig(paths=100_000, end_date=maturity, num_steps=251)
gbm = dp.GBMProcess(
    market_data=market,
    process_params=dp.GBMParams(initial_value=100.0, volatility=0.20),
    sim_config=sim_config,
)

mc_arith = dp.OptionValuation(
    underlying=gbm,
    spec=arith_call,
    pricing_method=dp.PricingMethod.MONTE_CARLO,
    params=dp.MonteCarloParams(random_seed=42),
)
print(f"Arithmetic Asian call (MC):  {mc_arith.present_value():.4f}")
print(f"Arithmetic Asian call (BSM): {bsm_arith.present_value():.4f}")

Arithmetic Asian call (MC):  5.7883
Arithmetic Asian call (BSM): 5.7789


## 5) Binomial Tree Pricing

In [9]:
binom_arith = dp.OptionValuation(
    underlying=underlying,
    spec=arith_call,
    pricing_method=dp.PricingMethod.BINOMIAL,
    params=dp.BinomialParams(num_steps=251, asian_tree_averages=400),
)
# higher number of averages gives more accurate price, but increases runtime

print(f"Arithmetic Asian call (Binomial): {binom_arith.present_value():.4f}")
print(f"Arithmetic Asian call (BSM):      {bsm_arith.present_value():.4f}")

Arithmetic Asian call (Binomial): 5.7880
Arithmetic Asian call (BSM):      5.7789


## 6) Puts

In [11]:
arith_put = dp.AsianSpec(
    averaging=dp.AsianAveraging.ARITHMETIC,
    exercise_type=dp.ExerciseType.EUROPEAN,
    option_type=dp.OptionType.PUT,
    strike=100.0,
    maturity=maturity,
    num_observations=252,
)

bsm_put = dp.OptionValuation(
    underlying=underlying, spec=arith_put, pricing_method=dp.PricingMethod.BSM
)
print(f"Arithmetic Asian call (BSM): {bsm_arith.present_value():.4f}")
print(f"Arithmetic Asian put  (BSM): {bsm_put.present_value():.4f}")

Arithmetic Asian call (BSM): 5.7789
Arithmetic Asian put  (BSM): 3.3606


## 7) Forward-Starting Asian

When `averaging_start` is set to a future date, the averaging period
begins later. The option is a forward-starting Asian.

In [12]:
fwd_call = dp.AsianSpec(
    averaging=dp.AsianAveraging.ARITHMETIC,
    exercise_type=dp.ExerciseType.EUROPEAN,
    option_type=dp.OptionType.CALL,
    strike=100.0,
    maturity=maturity,
    averaging_start=dt.datetime(2025, 12, 15),  # averaging starts in 6 months
    num_observations=126,  # ~half-year of daily steps
)

bsm_fwd = dp.OptionValuation(
    underlying=underlying, spec=fwd_call, pricing_method=dp.PricingMethod.BSM
)
print(f"Forward-starting Asian call (BSM): {bsm_fwd.present_value():.4f}")
print(f"Full-period Asian call (BSM):      {bsm_arith.present_value():.4f}")

Forward-starting Asian call (BSM): 8.3020
Full-period Asian call (BSM):      5.7789


## 8) Explicit Fixing Dates

Instead of uniform time steps, you can specify exact dates on which the
spot is observed for the average. This is common for OTC Asian options.

In [14]:
# Monthly fixing dates
fixing_dates = [dt.datetime(2025, m, 15) for m in range(7, 13)] + [
    dt.datetime(2026, m, 15) for m in range(1, 7)
]

fixing_call = dp.AsianSpec(
    averaging=dp.AsianAveraging.ARITHMETIC,
    exercise_type=dp.ExerciseType.EUROPEAN,
    option_type=dp.OptionType.CALL,
    strike=100.0,
    maturity=maturity,
    fixing_dates=fixing_dates,
)

# Explicit fixing dates require MC
mc_fixing = dp.OptionValuation(
    underlying=gbm,
    spec=fixing_call,
    pricing_method=dp.PricingMethod.MONTE_CARLO,
    params=dp.MonteCarloParams(random_seed=42),
)
print(f"Asian call with monthly fixings (MC): {mc_fixing.present_value():.4f}")

Asian call with monthly fixings (MC): 6.1967


## 9) Seasoned Asian (In-Progress)

A *seasoned* Asian option already has some observed fixings. The realised
average feeds into the payoff via Hull's $K^*$ adjustment.

In [15]:
seasoned_call = dp.AsianSpec(
    averaging=dp.AsianAveraging.ARITHMETIC,
    exercise_type=dp.ExerciseType.EUROPEAN,
    option_type=dp.OptionType.CALL,
    strike=100.0,
    maturity=maturity,
    num_observations=252,
    observed_average=98.5,  # average of 63 fixings already observed
    observed_count=63,
)

bsm_seasoned = dp.OptionValuation(
    underlying=underlying, spec=seasoned_call, pricing_method=dp.PricingMethod.BSM
)
print(f"Seasoned Asian call (BSM): {bsm_seasoned.present_value():.4f}")
print(f"Fresh Asian call (BSM):    {bsm_arith.present_value():.4f}")

Seasoned Asian call (BSM): 4.4644
Fresh Asian call (BSM):    5.7789


## 10) American Asian

American exercise is supported via MC (Longstaff-Schwartz) and binomial trees.

In [16]:
am_asian = dp.AsianSpec(
    averaging=dp.AsianAveraging.ARITHMETIC,
    exercise_type=dp.ExerciseType.AMERICAN,
    option_type=dp.OptionType.PUT,
    strike=100.0,
    maturity=maturity,
    num_observations=252,
)

mc_am = dp.OptionValuation(
    underlying=gbm,
    spec=am_asian,
    pricing_method=dp.PricingMethod.MONTE_CARLO,
    params=dp.MonteCarloParams(random_seed=42),
)
print(f"American Asian put (MC): {mc_am.present_value():.4f}")

American Asian put (MC): 4.0589


## 11) Method Comparison

In [17]:
print(f"{'Method':<40} {'Price':>8}")
print("-" * 50)
print(f"{'Arithmetic call — BSM analytical':<40} {bsm_arith.present_value():>8.4f}")
print(f"{'Arithmetic call — Monte Carlo':<40} {mc_arith.present_value():>8.4f}")
print(f"{'Arithmetic call — Binomial tree':<40} {binom_arith.present_value():>8.4f}")
print(f"{'Geometric call — BSM analytical':<40} {bsm_geo.present_value():>8.4f}")
print(f"{'Forward-starting call — BSM':<40} {bsm_fwd.present_value():>8.4f}")
print(f"{'Seasoned call — BSM':<40} {bsm_seasoned.present_value():>8.4f}")

Method                                      Price
--------------------------------------------------
Arithmetic call — BSM analytical           5.7789
Arithmetic call — Monte Carlo              5.7883
Arithmetic call — Binomial tree            5.7880
Geometric call — BSM analytical            5.5417
Forward-starting call — BSM                8.3020
Seasoned call — BSM                        4.4644
